# Nairobi Air Quality Data Wrangling with MongoDB

1. Setup

- from pprint import PrettyPrinter: alat untuk mencetak data bersarang (dictionary) dengan rapi.
- import pandas as pd: library pengolah data.
- pp = PrettyPrinter(indent=2): membuat printer dengan indentasi 2 spasi.

In [7]:
from pprint import PrettyPrinter
import pandas as pd

# Create a PrettyPrinter instance with "indent=2"
pp = PrettyPrinter(indent=2)

2. Memuat data ke MongoDB

- from pymongo import MongoClient: library untuk terhubung ke MongoDB.
- from load_mongo_data import load_nairobi_to_mongodb: fungsi pembantu untuk mengisi database.
- host="localhost", port=27017: alamat server MongoDB lokal.
- load_nairobi_to_mongodb(host=host): memasukkan data ke koleksi air-quality.nairobi. Hasilnya 10.000 dokumen berhasil dimuat.

In [8]:
# %pip install pymongo
from pymongo import MongoClient
from load_mongo_data import load_nairobi_to_mongodb
host="localhost"
port=27017

# Connect to MongoDB on localhost
load_nairobi_to_mongodb(host=host)

✅ Nairobi data successfully loaded to MongoDB at localhost:27017!
   Collection: air-quality.nairobi | Documents inserted: 10000


3. Menjelajahi database

- MongoClient(f"mongodb://{host}:{port}"): membuat koneksi ke server.
- client.list_database_names() dan pp.pprint(...): menampilkan daftar database (admin, air-quality, config, local).
- db = client["air-quality"]: memilih database kualitas udara.
- db.list_collection_names(): menampilkan koleksi, yaitu nairobi.
- nairobi = db["nairobi"]: memilih koleksi nairobi.

In [9]:
# Get list of databases
client = MongoClient(f"mongodb://{host}:{port}")

database_list = client.list_database_names()

# Print using pretty printer
pp.pprint(database_list)

['admin', 'air-quality', 'config', 'local']


In [10]:
# Access the air_quality database
db = client["air-quality"]

In [11]:
# Get list of collections
collection_list = db.list_collection_names()

# Print the collections
print(collection_list)

['nairobi']


In [12]:
# Access the nairobi collection
nairobi = db["nairobi"]

4. Memeriksa isi koleksi

- nairobi.count_documents({}): menghitung dokumen, ada 10.000.
- nairobi.find_one(): mengambil satu contoh dokumen, yang berisi location (lat, lon), sensor_id, timestamp, value, dan value_type.
- nairobi.distinct("sensor_id"): ada 10 sensor unik (nairobi_001 sampai nairobi_010).
- nairobi.distinct("value_type"): hanya satu jenis pengukuran, yaitu P2 (PM2.5).

In [13]:
# Count total documents
total_documents = nairobi.count_documents({})

print(f"Total documents: {total_documents:,}")

Total documents: 10,000


In [14]:
# Get one sample document
sample_document = nairobi.find_one()

# Print it
pp.pprint(sample_document)

{ '_id': ObjectId('6ad36e3f2d8dd1b7a0a5659f'),
  'location': {'lat': -1.2696061360394681, 'lon': 36.79165988896354},
  'sensor_id': 'nairobi_009',
  'timestamp': '2024-01-01 00:00:00',
  'value': 45.42,
  'value_type': 'P2'}


In [15]:
# Get distinct sensor IDs
unique_sensors = nairobi.distinct("sensor_id")

print(f"Number of unique sensors: {len(unique_sensors)}")
print(f"First 5 sensors: {unique_sensors[:5]}")

Number of unique sensors: 10
First 5 sensors: ['nairobi_001', 'nairobi_002', 'nairobi_003', 'nairobi_004', 'nairobi_005']


In [16]:
measurement_types = nairobi.distinct("value_type")
print(f"Measurement types: {measurement_types}")

Measurement types: ['P2']


5. Agregasi: jumlah bacaan per sensor

- $group dengan {"_id": "$sensor_id", "count": {"$sum": 1}}: menghitung bacaan tiap sensor.
- $sort (count: -1) dan $limit 5: 5 sensor dengan bacaan terbanyak. Tertinggi nairobi_008 (1.051), lalu nairobi_004 (1.047) dan nairobi_010 (1.033).
- nairobi.aggregate(pipeline): menjalankan pipeline.

In [17]:
# Create aggregation pipeline
pipeline = [
    {"$group": {"_id": "$sensor_id", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 5}
]

# Execute aggregation
sensor_reading_counts = list(nairobi.aggregate(pipeline))

# Print results
pp.pprint(sensor_reading_counts)

[ {'_id': 'nairobi_008', 'count': 1051},
  {'_id': 'nairobi_004', 'count': 1047},
  {'_id': 'nairobi_010', 'count': 1033},
  {'_id': 'nairobi_007', 'count': 1032},
  {'_id': 'nairobi_002', 'count': 1009}]


6. Agregasi untuk satu sensor

- first_sensor = unique_sensors[0] (dengan cek if unique_sensors): memilih sensor pertama, nairobi_001.
- $match lalu $group: menghitung bacaan sensor tersebut, hasilnya 953.

In [18]:
# Get the first sensor ID
if unique_sensors:
    first_sensor = unique_sensors[0]
else:
    print("No sensors found!")

# Create pipeline to count by type for this sensor
pipeline = [
    {"$match": {"sensor_id": first_sensor}},
    {"$group": {"_id": "$sensor_id", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}}
]

sensor_type_counts = list(nairobi.aggregate(pipeline))
pp.pprint(sensor_type_counts)

[{'_id': 'nairobi_001', 'count': 953}]


7. Query bacaan PM2.5

- nairobi.find({"sensor_id": first_sensor, "value_type": "P2"}, projection={...}).limit(5): mengambil 5 bacaan pertama dengan hanya kolom timestamp dan value, tanpa _id.
- Hasilnya berupa bacaan sekitar tiap 7 jam, dengan nilai 27,75 sampai 39,99.

In [19]:
# Query PM2.5 readings from first sensor
pm25_cursor = nairobi.find(
    {"sensor_id": first_sensor, "value_type": "P2"},
    projection={"timestamp": True, "value": True, "_id": False}
).limit(5)

# Convert to list and print
pm25_sample = list(pm25_cursor)
pp.pprint(pm25_sample)

[ {'timestamp': '2024-01-01 09:00:00', 'value': 27.75},
  {'timestamp': '2024-01-01 16:00:00', 'value': 33.55},
  {'timestamp': '2024-01-01 23:00:00', 'value': 34.51},
  {'timestamp': '2024-01-02 09:00:00', 'value': 32.2},
  {'timestamp': '2024-01-02 16:00:00', 'value': 39.99}]


8. Membuat DataFrame

- pd.DataFrame(list(nairobi.find(...))): mengubah hasil query menjadi tabel.
- .assign(timestamp=pd.to_datetime(...)): mengubah teks waktu menjadi tipe datetime.
- .set_index("timestamp") dan .sort_index(): waktu sebagai indeks, diurutkan kronologis.
- df_raw.head() dan df_raw.shape: hasilnya (953, 1).

In [20]:
# Convert to DataFrame
df_raw = (
    pd.DataFrame(list(nairobi.find(
        {"sensor_id": first_sensor, "value_type": "P2"},
        projection={"timestamp": True, "value": True, "_id": False}
    )))
    .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
    .set_index("timestamp")
    .sort_index()
)

print(df_raw.head())
print(f"\nShape: {df_raw.shape}")

                     value
timestamp                 
2024-01-01 09:00:00  27.75
2024-01-01 16:00:00  33.55
2024-01-01 23:00:00  34.51
2024-01-02 09:00:00  32.20
2024-01-02 16:00:00  39.99

Shape: (953, 1)


9. Cek kualitas data

- df_raw.isnull().sum(): tidak ada nilai kosong.
- df_raw.describe(): rata-rata PM2.5 sekitar 39,76, median 39,89, minimum 8,03, maksimum 67,41, simpangan baku 9,82.

In [21]:
# Check for missing values
print("Missing values:")
print(df_raw.isnull().sum())

print("\nBasic statistics:")
print(df_raw.describe())

Missing values:
value    0
dtype: int64

Basic statistics:
            value
count  953.000000
mean    39.755792
std      9.816934
min      8.030000
25%     33.100000
50%     39.890000
75%     46.360000
max     67.410000


10. Fungsi wrangle_data

- Parameter db, collection, host, port=27017: dapat dipakai ulang untuk database lain.
- .find({"value_type": "P2"}, projection=...) dan .sort("timestamp", 1): mengambil semua bacaan PM2.5 diurutkan menurut waktu.
- .assign(timestamp=pd.to_datetime(...)): konversi ke datetime.
- .dt.tz_localize("UTC").dt.tz_convert("Africa/Nairobi"): menandai waktu sebagai UTC lalu mengubahnya ke zona waktu Nairobi (UTC+3).
- .set_index("timestamp"), .dropna(), .sort_index(): indeks waktu, buang nilai kosong, urutkan.
- .rename(columns={"value": "pm25"}): mengganti nama kolom agar jelas.
- Hasil uji: 10.000 baris, 1 kolom pm25, dengan waktu sudah dalam zona +03:00.

In [23]:
db = "air-quality"
collection = "nairobi"
host = "localhost"

def wrangle_data(
    db, 
    collection, 
    host=host, 
    port=27017
    ):
    """
    Retrieve and clean Nairobi air quality data from MongoDB.
    
    Parameters:
    -----------
    host : str
        MongoDB host address (default: localhost)
    port : int
        MongoDB port (default: 27017)
    
    Returns:
    --------
    pd.DataFrame
        Clean time series data indexed by timestamp with 'pm25' column
    """
    from pymongo import MongoClient
    
    # Connect to MongoDB
    client = MongoClient(f"mongodb://{host}:{port}")
    
    # Query and clean data using method chaining
    return (
        pd.DataFrame(
            list(
                client[db][collection]
                .find({"value_type": "P2"}, 
                      projection={"value": 1, "timestamp": 1, "_id": 0})
                .sort("timestamp", 1)
            )
        )
        # Convert timestamp to datetime
        .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
        # Set timezone (data is in UTC, convert to Nairobi time)
        .assign(timestamp=lambda x: x["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Africa/Nairobi"))
        # Set index
        .set_index("timestamp")
        # Handle missing values
        .dropna()
        # Sort chronologically
        .sort_index()
        # Rename column for clarity
        .rename(columns={"value": "pm25"})
    )
    


# Test the function
df_clean = wrangle_data(db, collection, host)
print(df_clean.head())
print(f"\nShape: {df_clean.shape}")

                            pm25
timestamp                       
2024-01-01 03:00:00+03:00  45.42
2024-01-01 04:00:00+03:00  28.71
2024-01-01 05:00:00+03:00  30.34
2024-01-01 06:00:00+03:00  28.35
2024-01-01 07:00:00+03:00  52.78

Shape: (10000, 1)


11. Menyimpan fungsi ke modul

- %%writefile mongo_wrangle.py: menyimpan fungsi wrangle_data ke file mongo_wrangle.py (dengan docstring dan import) supaya bisa diimpor di notebook berikutnya.

In [24]:
%%writefile mongo_wrangle.py
"""
MongoDB Data Wrangling Module for Nairobi Air Quality Data

This module provides functions to retrieve and clean air quality data
from MongoDB for time series analysis.
"""

import pandas as pd
from pymongo import MongoClient


def wrangle_data(
    db, 
    collection, 
    host, 
    port=27017
    ):
    
    """
    Retrieve and clean Nairobi air quality data from MongoDB.
    
    Parameters:
    -----------
    host : str
        MongoDB host address (default: localhost)
    port : int
        MongoDB port (default: 27017)
    
    Returns:
    --------
    pd.DataFrame
        Clean time series data indexed by timestamp with 'pm25' column
    """
    # Connect to MongoDB
    client = MongoClient(f"mongodb://{host}:{port}")
    
    # Query and clean data using method chaining
    return(
        pd.DataFrame(
            list(
                client[db][collection]
                .find({"value_type": "P2"}, 
                      projection={"value": 1, "timestamp": 1, "_id": 0})
                .sort("timestamp", 1)
            )
        )
        # Convert timestamp to datetime
        .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
        # Set timezone (data is in UTC, convert to Nairobi time)
        .assign(timestamp=lambda x: x["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Africa/Nairobi"))
        # Set index
        .set_index("timestamp")
        # Handle missing values
        .dropna()
        # Sort chronologically
        .sort_index()
        # Rename column for clarity
        .rename(columns={"value": "pm25"})
    )

Overwriting mongo_wrangle.py


Kesimpulan singkat

- Notebook ini menunjukkan cara mengambil dan menyiapkan data kualitas udara dari MongoDB. 
- Data PM2.5 Nairobi (10.000 dokumen dari 10 sensor) dijelajahi lewat count_documents, distinct, find, dan pipeline agregasi. 
- Pada satu sensor, 953 bacaan tidak memiliki nilai kosong dan rata-rata PM2.5 sekitar 39,8. 
- Seluruh langkah pembersihan (query, konversi waktu, zona waktu Nairobi, penghapusan nilai kosong, penamaan kolom) dibungkus dalam fungsi wrangle_data() dan disimpan sebagai mongo_wrangle.
- py. Hasilnya adalah deret waktu pm25 yang bersih dan siap dianalisis atau diramal pada tahap berikutnya.

Catatan: fungsi wrangle_data mengambil data dari semua sensor sekaligus, jadi bacaan antar sensor bercampur dalam satu deret waktu. Untuk peramalan yang lebih akurat, mungkin perlu dirata-ratakan per jam atau difilter per sensor. Selain itu, sel “count by type” sebenarnya menghitung bacaan per sensor, bukan per tipe.